In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Conv2D, Reshape
from tensorflow.keras.models import Sequential
print("All modules imported successfully!")

In [ ]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, emb, latent, heads=6, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb
        self.latent = latent
        self.heads = heads

        # 1. Separate projections for Q, K, and V for all heads combined
        self.wq = Dense(self.latent * self.heads)
        self.wk = Dense(self.latent * self.heads)
        self.wv = Dense(self.latent * self.heads)
        
        # 2. Final projection to ensure the output matches 'emb' size for the residual connection
        self.dense_proj = Dense(self.emb)

        self.dense = Sequential([
            Dense(latent, activation='relu'),
            Dense(emb)
        ])
        
        # Initialize Layer Normalization blocks
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def pos_enc(self, duncan_seq_len):
        pos = tf.cast(tf.range(duncan_seq_len), tf.float32)[:, tf.newaxis]
        dimensions = tf.cast(tf.range(self.emb), tf.float32)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2 * (dimensions // 2)) / tf.cast(self.emb, tf.float32))
        angle_rads = pos * angle_rates
        
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        
        pos_encoding = tf.stack([sines, cosines], axis=-1)
        pos_encoding = tf.reshape(pos_encoding, [duncan_seq_len, self.emb])
        return pos_encoding

    def split_heads(self, x, batch_size):
        """
        Splits the last dimension into (heads, latent).
        Transposes the result so the shape is (batch_size, heads, seq_len, latent)
        """
        x = tf.reshape(x, (batch_size, -1, self.heads, self.latent))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs, mask=None):
        """
        inputs: (batch_size, duncan_seq_len, emb)
        """
        bs = tf.shape(inputs)[0] # Dynamic batch size
        sl = tf.shape(inputs)[1] # Dynamic sequence length
        
        # 1. Add Positional Encoding
        pos_enc = self.pos_enc(sl) 
        x = inputs + pos_enc # (batch_size, seq_len, emb)

        # 2. Linear Projections
        q = self.wq(x)  # (batch_size, seq_len, latent * heads)
        k = self.wk(x)  # (batch_size, seq_len, latent * heads)
        v = self.wv(x)  # (batch_size, seq_len, latent * heads)

        # 3. Split into Multi-Heads
        q = self.split_heads(q, bs) # (batch_size, heads, seq_len, latent)
        k = self.split_heads(k, bs) # (batch_size, heads, seq_len, latent)
        v = self.split_heads(v, bs) # (batch_size, heads, seq_len, latent)

        # 4. Scaled Dot-Product Attention
        # transpose_b=True swaps the last two dimensions of K -> (batch_size, heads, latent, seq_len)
        attention_scores = tf.matmul(q, k, transpose_b=True) # (batch_size, heads, seq_len, seq_len)
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))

        if mask is not None:
            # Multiply the mask by a huge negative number and add it to the scores.
            # Assuming duncan_mask has 1s for pads and 0s for real tokens.
            attention_scores += (mask * -1e9)
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        
        # Multiply weights by V
        attention_output = tf.matmul(attention_weights, v) # (batch_size, heads, seq_len, latent)

        # 5. Recombine Heads
        # Transpose back to (batch_size, seq_len, heads, latent)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        # Flatten the last two dimensions
        concat_attention = tf.reshape(attention_output, (bs, sl, self.latent * self.heads)) # (batch_size, seq_len, latent * heads)

        # 6. Final Linear Projection and First Residual
        proj_attention = self.dense_proj(concat_attention) # (batch_size, seq_len, emb)
        out1 = self.layernorm1(proj_attention + x) # Residual connection now works perfectly

        # 7. Feed-Forward Network and Second Residual
        ffn_output = self.dense(out1) # (batch_size, seq_len, emb)
        encoder_output = self.layernorm2(ffn_output + out1) # (batch_size, seq_len, emb)

        return encoder_output

In [ ]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, emb, latent, heads=6, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb
        self.latent = latent
        self.heads = heads

        # 1. Weights for Masked Self-Attention
        self.wq_self = Dense(self.latent * self.heads)
        self.wk_self = Dense(self.latent * self.heads)
        self.wv_self = Dense(self.latent * self.heads)
        self.dense_proj_self = Dense(self.emb)

        # 2. Weights for Cross-Attention (Encoder-Decoder)
        self.wq_cross = Dense(self.latent * self.heads)
        self.wk_cross = Dense(self.latent * self.heads)
        self.wv_cross = Dense(self.latent * self.heads)
        self.dense_proj_cross = Dense(self.emb)

        # 3. Feed-Forward Network
        self.dense = Sequential([
            Dense(latent, activation='relu'),
            Dense(emb)
        ])
        
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.heads, self.latent))
        return tf.transpose(x, perm=[0, 2, 1, 3])
    
    def pos_enc(self, duncan_seq_len, emb):
        pos = tf.cast(tf.range(duncan_seq_len), tf.float32)[:, tf.newaxis]
        dimensions = tf.cast(tf.range(emb), tf.float32)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2 * (dimensions // 2)) / tf.cast(emb, tf.float32))
        angle_rads = pos * angle_rates
        
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        
        pos_encoding = tf.stack([sines, cosines], axis=-1)
        pos_encoding = tf.reshape(pos_encoding, [duncan_seq_len, emb])
        return pos_encoding
    
    def call(self, inputs, enc_output, look_ahead_mask=None, duncan_padding_mask=None):
        """
        inputs: (batch_size, tar_seq_len, emb)
        enc_output: (batch_size, inp_seq_len, emb)
        """
        # Using tf.shape()[index] is safer for dynamic batch sizing in graph mode
        bs = tf.shape(inputs)[0] 
        sl = tf.shape(inputs)[1] 

        # --- BLOCK 1: MASKED SELF-ATTENTION ---
        pos_enc = self.pos_enc(sl, self.emb)
        x = inputs + pos_enc

        q = self.wq_self(x) 
        k = self.wk_self(x) 
        v = self.wv_self(x) 

        q = self.split_heads(q, bs) 
        k = self.split_heads(k, bs) 
        v = self.split_heads(v, bs) 

        attention_scores = tf.matmul(q, k, transpose_b=True) 
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))

        # Apply the Look-Ahead Mask (Autoregressive feature)
        if look_ahead_mask is not None:
            attention_scores += (look_ahead_mask * -1e9)
            
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, v) 
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (bs, sl, self.latent * self.heads)) 
        
        proj_attention = self.dense_proj_self(concat_attention) 
        proj_attention = self.layernorm1(proj_attention + x) 

        # --- BLOCK 2: CROSS-ATTENTION ---
        # Keys and Values come from the Encoder's output
        enc_k = self.wk_cross(enc_output)  
        enc_v = self.wv_cross(enc_output)  
        
        # Query comes from the previous Decoder block
        dec_q = self.wq_cross(proj_attention)  
        
        dec_q = self.split_heads(dec_q, bs) 
        enc_k = self.split_heads(enc_k, bs) 
        enc_v = self.split_heads(enc_v, bs) 

        cross_attention_scores = tf.matmul(dec_q, enc_k, transpose_b=True) 
        cross_attention_scores = cross_attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))
        
        # Apply the Padding Mask (Ignore padded encoder tokens)
        if duncan_padding_mask is not None:
            cross_attention_scores += (duncan_padding_mask * -1e9)

        cross_attention_weights = tf.nn.softmax(cross_attention_scores, axis=-1)
        cross_attention_output = tf.matmul(cross_attention_weights, enc_v) 
        cross_attention_output = tf.transpose(cross_attention_output, perm=[0, 2, 1, 3])
        concat_cross_attention = tf.reshape(cross_attention_output, (bs, sl, self.latent * self.heads)) 
        
        proj_cross_attention = self.dense_proj_cross(concat_cross_attention) 
        proj_cross_attention = self.layernorm2(proj_cross_attention + proj_attention) 

        # --- BLOCK 3: FEED-FORWARD NETWORK ---
        ffn_output = self.dense(proj_cross_attention) 
        decoder_output = self.layernorm3(ffn_output + proj_cross_attention) 
        
        return decoder_output

In [ ]:
class PatchEmbedding(tf.keras.layers.Layer):
    def __init__(self, image_size=256, patch_size=16, emb_dim=512, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.patch_size = patch_size
        self.emb_dim = emb_dim
        self.num_patches = (image_size // patch_size) ** 2

        self.projection = Conv2D(filters=emb_dim, kernel_size=patch_size,strides=patch_size, padding='valid')

        self.flatten_sequence = Reshape((self.num_patches, self.emb_dim))
        self.position_emb = tf.Variable(
            initial_value=tf.random.normal([1, self.num_patches, self.emb_dim], stddev=0.02), trainable=True
        )

    def call(self, images):
        patches = self.projection(images)
        seq = self.flatten_sequence(patches)
        encoded_seq = seq + self.position_emb
        return encoded_seq

In [ ]:
class ViT_Captioning_Transformer(tf.keras.Model):
    def __init__(self, num_layers, emb, latent, heads, target_vocab_size, **kwargs):
        super().__init__(**kwargs)
        
        # The new Vision front-end
        self.image_patcher = PatchEmbedding(image_size=256, patch_size=16, emb_dim=emb)
        
        # Your custom blocks (Make sure Encoder no longer adds pos_enc)
        self.encoder = [Encoder(emb, latent, heads) for _ in range(num_layers)]
        self.decoder = [Decoder(emb, latent, heads) for _ in range(num_layers)]
        
        self.final_layer = Dense(target_vocab_size)

    def call(self, image_inputs, targets, training=True, look_ahead_mask=None, dec_padding_mask=None):
        
        # 1. Turn the image into a sequence of positional patches
        # Output shape: (batch_size, 256, emb)
        enc_output = self.image_patcher(image_inputs)
        
        # 2. Pass through Encoders
        for enc in self.encoder:
            # Note: We don't need an enc_padding_mask here because image patches 
            # are perfectly square and never padded with zeros!
            enc_output = enc(enc_output, mask=None) 
            
        # 3. Pass through Decoders
        dec_output = targets
        for dec in self.decoder:
            dec_output = dec(
                dec_output, 
                enc_output, 
                look_ahead_mask=look_ahead_mask, 
                duncan_padding_mask=dec_padding_mask
            )
            
        final_output = self.final_layer(dec_output)
        return final_output